In [ ]:
# 파이토치 라이브러리 호출
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# CNN + 미니배치 + 사전학습 ResNet50 전이학습

# 실습에 필요한 데이터셋
from torchvision import datasets
raw_train = datasets.SVHN( root='./data', split='train', download=True )
raw_test = datasets.SVHN( root='./data', split='test', download=True )

from torch.nn.modules import Sequential
# Pytorch 의 tensor type 으로 변경하고 0~255(밝기정도) 값들 0 ~ 1 정규화
# 샘플로 2만개만 사용
# .to(device) : 연산속도를 위해 CPU 아닌 GPU 처리
# 0~255(밝기정도) 값들 0 ~ 1 정규화 - 단,Y 타겟/정답 제외
X_train = torch.tensor(raw_train.data[:20000], dtype=torch.float32).to(device) / 255.0
Y_train = torch.tensor(raw_train.labels[:20000], dtype=torch.long).to(device)
X_test = torch.tensor(raw_test.data[:2000], dtype=torch.float32).to(device) / 255.0
Y_test = torch.tensor(raw_test.labels[:2000], dtype=torch.long).to(device)

#####################  미니배치  #####################
from torch.utils.data import TensorDataset, DataLoader
# 특성과 정답 1:1 매핑하여 묶음
train_dataset = TensorDataset(X_train, Y_train)
# 묶은 데이터셋을 지정한 배치 크기 단위로 자르고 셔플로 랜덤 병렬 로딩
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
#####################  미니배치  #####################



#################### 사전학습모델 ####################
# 기존 CNN 설정 지우고 사전학습으로 교체
import torch.nn as nn
import torchvision.models as models
# 사전학습 모델에 저장된 가중치 기본값 불러오기
model = models.resnet50( weights='DEFAULT')
# 사전모델 학습의 특성맵 수
# 다음 레이어 연결시 이전 레이어의 특성맵 수 알아야 함.
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear( in_features = in_features , out_features=128 ),
    # 과적합방지
    nn.ReLU(),
    nn.Dropout( 0.2 ),
    # 출력층
    nn.Linear( in_features = 128 , out_features = 10 )
)
# 모델 전체를 GPU(device)로 이동
model = model.to(device)
#################### 사전학습모델 ####################


# 3) 손실함수 : 다중 분류 - softmax
criterion = nn.CrossEntropyLoss()

# 4) 옵티마이저 ; optimizer = optim.AdamW( model.parameters(), lr=학습률)
import torch.optim as optim
optimizer = optim.AdamW( model.parameters(), lr = 0.0001 )



# 5) 에포크(학습수) 단위 학습( 순전파 -> 오차계산 -> 역전파 -> 업데이트 )
for epoch in range( 5 ) :
    # 학습모드
    model.train()
#####################  미니배치  #####################
    total_loss = 0.0
    for batch_X , batch_Y in train_loader :
        # 이전 기울기 초기화
        optimizer.zero_grad()
        # 순전파(학습)
        logits = model(batch_X)
        # 오차(학습과 정답 차이)
        loss = criterion(logits , batch_Y)
        # 역전파(오차 이용한 기울기/미분 가중치/편향 재설정)
        loss.backward()
        # 가중치/편향 업데이트
        optimizer.step()
        #
        total_loss += loss.item()
    # 평균오차
    total_loss = total_loss / len(train_loader)
    if epoch % 1 == 0 :
        print(epoch , loss.item())
#####################  미니배치  #####################


# 6) 모델 평가. model.train()학습모드 , model.eval() 평가/추론모드
model.eval()
with torch.no_grad() :
    # 테스트용 학습하기( 기울기 업데이트 안 함 )
    test_logits = model(X_test)
    # 확률중 가장 높은 값 추출 ( 소프트맥스 )
    test_pred = torch.argmax(test_logits, dim=1)

from sklearn.metrics import accuracy_score
print( accuracy_score( Y_test.cpu().numpy() , test_pred.cpu().numpy() ) )

# 7) 모델 추론 # [ [ 공부시간,결석,G1,G2 ] ]

# 모델 가중치 저장
# 머신러닝 pkl , 딥러닝가중치 pth , 딥러닝모델 onnx  =>  스프링실행 가능
# 실무 
torch.save(model.state_dict(), 'resnet50_custom.pth')